## Dataset Overview:
The Heart Disease dataset contains 14 attributes and is used to predict the presence of heart disease in patients. The 

attributes include:

- **age:** Age of the patient
- **sex:** Gender of the patient (1 = male, 0 = female)
- **cp:** Chest pain type (0 to 3)
- **trestbps:** Resting blood pressure (in mm Hg)
- **chol:** Serum cholesterol in mg/dl
- **fbs:** Fasting blood sugar > 120 mg/dl (1 = true; 0 = false)
- **restecg:** Resting electrocardiographic results (0 to 2)
- **thalach:** Maximum heart rate achieved
- **exang:** Exercise-induced angina (1 = yes; 0 = no)
- **oldpeak:** ST depression induced by exercise relative to rest
- **slope:** The slope of the peak exercise ST segment
- **ca:** Number of major vessels (0–3) colored by fluoroscopy
- **thal:** Thalassemia (3 = normal; 6 = fixed defect; 7 = reversible defect)
- **target:** Diagnosis of heart disease (1 = disease present, 0 = no disease)

In [13]:
# Import necessary libraries
import pandas as pd

# Step 1: Load the Dataset from the URL with defining the column names as the dataset doesn’t have headers.
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
df = pd.read_csv(url, names=columns)

# Show the first 5 rows to verify the data 
print(df.head())

# Print the information of dataset
df.info()


    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0  0.0      2.0    172.0    0.0      1.4   

   slope   ca thal  target  
0    3.0  0.0  6.0       0  
1    2.0  3.0  3.0       2  
2    2.0  2.0  7.0       1  
3    3.0  0.0  3.0       0  
4    1.0  0.0  3.0       0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    float64
 4   chol      303 non-n

In [14]:
# Check for missing values
print(df.isnull().sum())


age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64


In [18]:
# Step 3: Handle Missing Values
# Replace missing values denoted by '?' with NaN
df.replace('?', pd.NA, inplace=True)

# Convert 'ca' and 'thal' columns to numeric (since they had '?' values initially)
df['ca'] = pd.to_numeric(df['ca'], errors='coerce')
df['thal'] = pd.to_numeric(df['thal'], errors='coerce')

# Now check again for missing values after '?' replacement
print(df.isnull().sum())

# Fill missing values for 'ca' and 'thal' with the mode (most frequent value)
df['ca'].fillna(df['ca'].mode(), inplace=True)
df['thal'].fillna(df['ca'].mode(), inplace=True)

# Check for any remaining missing values
print(df.isnull().sum())

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64


In [19]:
# Step 4: Create New Columns
# Create a new column 'cholesterol_level' based on the 'chol' value
def cholesterol_level(chol):
    if chol > 240:
        return 'High'
    elif 200 < chol <= 240:
        return 'Borderline'
    else:
        return 'Normal'

# Apply the function to create the new column
df['cholesterol_level'] = df['chol'].apply(cholesterol_level)
df['cholesterol_level']

0      Borderline
1            High
2      Borderline
3            High
4      Borderline
          ...    
298          High
299        Normal
300        Normal
301    Borderline
302        Normal
Name: cholesterol_level, Length: 303, dtype: object

In [20]:
# Create a new column 'age_group' to categorize patients into age groups (almost similar to the step 4)
def age_group(age):
    if age > 40:
        return 'Young'
    elif 40 < age <= 60:
        return 'Midle-aged'
    else:
        return 'Senior'

# Apply the function to create the new column
df['age_group'] = df['age'].apply(age_group)
df['age_group']


0       Young
1       Young
2       Young
3      Senior
4       Young
        ...  
298     Young
299     Young
300     Young
301     Young
302    Senior
Name: age_group, Length: 303, dtype: object

In [21]:
# Step 5: Analyze the Data
# Calculate average cholesterol levels by gender and age group
avg_chol_by_gender_age = df.groupby(['sex', 'age_group'])['chol'].mean().reset_index()
print("Average cholesterol by gender and age group:")
print(avg_chol_by_gender_age)

Average cholesterol by gender and age group:
   sex age_group        chol
0  0.0    Senior  205.400000
1  0.0     Young  264.815217
2  1.0    Senior  218.692308
3  1.0     Young  241.010363


In [22]:
# Find the number of patients with heart disease (target = 1) in each age group
heart_disease_by_age = df[df['target'] == 1].groupby('age_group')['target'].count().reset_index()
print("Number of patients with heart disease in each age group:")
print(heart_disease_by_age)

Number of patients with heart disease in each age group:
  age_group  target
0    Senior       3
1     Young      52


In [23]:
# Step 6: Export the Cleaned Data
# Write the cleaned and transformed data to a new CSV file
df.to_csv('cleaned_heart-disease.csv', index=False)


In [24]:
# Print a message to indicate successful completion
print("Data wrangling complete. Cleaned dataset saved as 'cleaned_heart-disease.csv'. ")

Data wrangling complete. Cleaned dataset saved as 'cleaned_heart-disease.csv'. 
